<a href="https://colab.research.google.com/github/jihandong/MLCExercises/blob/main/mlc_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Compiler Exercises

## Install the Environment




In [ ]:
%pip install -q "apache-tvm==0.26.0" "apache-tvm-ffi==0.1.16"

In [ ]:
import numpy as np
import tvm
from tvm.script import tirx as T

dtype = "float32"
a_np = np.random.rand(128, 128).astype(dtype)
b_np = np.random.rand(128, 128).astype(dtype)
c_mm_relu = np.maximum(a_np @ b_np, 0)

@tvm.script.ir_module
class MyModule:
    @T.prim_func(s_tir=True)
    def mm_relu(A: T.Buffer((128, 128), "float32"),
                B: T.Buffer((128, 128), "float32"),
                C: T.Buffer((128, 128), "float32")):
        T.func_attr({"global_symbol": "mm_relu", "tirx.noalias": True})
        Y = T.sblock_alloc_buffer((128, 128), dtype="float32")
        for i, j, k in T.grid(128, 128, 128):
            with T.sblock("Y"):
                vi, vj, vk = T.axis.remap("SSR", [i, j, k])
                with T.init():
                    Y[vi, vj] = T.float32(0)
                Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
        for i, j in T.grid(128, 128):
            with T.sblock("C"):
                vi, vj = T.axis.remap("SS", [i, j])
                C[vi, vj] = T.max(Y[vi, vj], T.float32(0))


## 2.4.5.1 J1 Profile

In [ ]:
from statistics import median


def transform(mod, j_factor):
    """Apply the loop transform from section 2.4.5.1 to a fresh schedule."""
    schedule = tvm.s_tir.Schedule(mod)
    matmul = schedule.get_sblock("Y", func_name="mm_relu")
    relu = schedule.get_sblock("C", func_name="mm_relu")
    _, j, k = schedule.get_loops(matmul)
    j_outer, j_inner = schedule.split(j, factors=[None, j_factor])
    schedule.reorder(j_outer, k, j_inner)
    schedule.reverse_compute_at(relu, j_outer)
    return schedule.mod


dev = tvm.cpu(0)
a_nd = tvm.runtime.tensor(a_np, device=dev)
b_nd = tvm.runtime.tensor(b_np, device=dev)
c_nd = tvm.runtime.empty(c_mm_relu.shape, dtype, dev)


def profile_ms(mod):
    """Return the median time per kernel call in milliseconds."""
    lib = tvm.build(mod, target="llvm")
    timer = lib.time_evaluator("mm_relu", dev, number=100, repeat=5)
    return median(timer(a_nd, b_nd, c_nd).results) * 1_000


In [ ]:
j_factors = [1,2,4,8,16,32,64,128]

baseline_ms = profile_ms(MyModule)
print(f"{'j_factor':>10} {'median (ms)':>14} {'speedup':>10}")
print(f"{'baseline':>10} {baseline_ms:14.4f} {'1.00x':>10}")

for j_factor in j_factors:
    scheduled = transform(MyModule, j_factor)
    elapsed_ms = profile_ms(scheduled)
    print(f"{j_factor:10d} {elapsed_ms:14.4f} {baseline_ms / elapsed_ms:9.2f}x")


  j_factor    median (ms)    speedup
  baseline         3.7445      1.00x
         1         2.9627      1.26x
         2         1.7178      2.18x
         4         0.7911      4.73x
         8         0.4841      7.73x
        16         0.3321     11.28x
        32         0.2881     13.00x
        64         0.3100     12.08x
       128         0.3085     12.14x


## 2.5 Exercises

In [ ]:
import IPython
import numpy as np
import tvm
from tvm.ir.module import IRModule
from tvm.script import tirx as T

def build_and_check(mod, symbol, arrays, expected, device):
    lib = tvm.build(mod, target="llvm")
    inputs = tuple(tvm.runtime.tensor(array, device=device) for array in arrays)
    out = tvm.runtime.empty(expected.shape, str(expected.dtype), device)
    args = (*inputs, out)
    lib[symbol](*args)
    try:
        if np.issubdtype(expected.dtype, np.integer):
            np.testing.assert_array_equal(out.numpy(), expected)
        else:
            np.testing.assert_allclose(out.numpy(), expected, rtol=1e-4, atol=1e-4)
    except AssertionError:
        print(f"{symbol}: FAIL")
        raise
    print(f"{symbol}: PASS")
    return lib, args

exercise_cpu = tvm.cpu(0)

### 2.5.1.2 Broadcast Add

In [ ]:
add_a = np.arange(16, dtype="int64").reshape(4, 4)
add_b = np.arange(4, 0, -1, dtype="int64")
add_expected = add_a + add_b

@tvm.script.ir_module
class MyAdd:
    @T.prim_func(s_tir=True)
    def add(A: T.Buffer((4, 4), "int64"),
            B: T.Buffer((4,), "int64"),
            C: T.Buffer((4, 4), "int64")):
        T.func_attr({"global_symbol": "add", "tirx.noalias": True})
        for i, j in T.grid(4, 4):
            with T.sblock("C"):
                vi = T.axis.spatial(4, i)
                vj = T.axis.spatial(4, j)
                C[vi, vj] = A[vi, vj] + B[vj]

_ = build_and_check(MyAdd, "add", (add_a, add_b), add_expected, exercise_cpu)


add: PASS


### 2.5.1.3 2D Convolution

Use NCHW layout, stride 1 and no padding. Two
3 × 3 filters give an output of shape `(1, 2,
6, 6)`.

$
\operatorname{Conv}[b,k,i,j]
= \sum_{d_i,d_j,q}
A\!\left[b,q,\mathrm{strides}\cdot i+d_i,\mathrm{strides}\cdot j+d_j\right]
\cdot W\!\left[k,q,d_i,d_j\right]
$

In [ ]:
conv_data = np.arange(64, dtype="int64").reshape(1, 1, 8, 8)
conv_weight = np.arange(18, dtype="int64").reshape(2, 1, 3, 3)
conv_windows = np.lib.stride_tricks.sliding_window_view(
    conv_data, (3, 3), axis=(-2, -1),
)
conv_expected = np.einsum("nchwij,ocij->nohw", conv_windows, conv_weight)


@tvm.script.ir_module
class MyConv:
    @T.prim_func(s_tir=True)
    def conv(A: T.Buffer((1, 1, 8, 8), "int64"),
             W: T.Buffer((2, 1, 3, 3), "int64"),
             C: T.Buffer((1, 2, 6, 6), "int64")):
        T.func_attr({"global_symbol": "conv", "tirx.noalias": True})
        # XXX: fix and duplicate dims are bad, single definition is better.
        for b, oc, ic, h, w, kh, kw in T.grid(1, 2, 1, 6, 6, 3, 3):
            with T.sblock("C"):
                 vb, voc, vh, vw, vic, vkh, vkw = T.axis.remap(
                     "SSSSRRR", [b, oc, h, w, ic, kh, kw])
                 with T.init():
                    C[vb, voc, vh, vw] = T.int64(0)
                 C[vb, voc, vh, vw] +=\
                    A[vb, vic, vh + vkh, vw + vkw] * W[voc, vic, vkh, vkw]

_ = build_and_check(
    MyConv, "conv", (conv_data, conv_weight), conv_expected, exercise_cpu,
)


conv: PASS


### 2.5.2.2 Batch Matmul + ReLU

Write the baseline for 16 independent 128 × 128 matrix products, then use the [book’s target schedule](https://book.mlc.ai/chapter_tensor_program/tensorir_exercises.html#exercise-3-transform-a-batch-matmul-program) to guide your transformations.

#### Example: reverse_compute_at

![reverse_compute_at example](https://raw.githubusercontent.com/jihandong/MLCExercises/main/assets/reverse_compute_at.jpg)

In [ ]:
bmm_rng = np.random.default_rng(0)
bmm_a = bmm_rng.standard_normal((16, 128, 128), dtype=np.float32)
bmm_b = bmm_rng.standard_normal((16, 128, 128), dtype=np.float32)
bmm_expected = np.maximum(bmm_a @ bmm_b, 0)

# NOTE: a baseline to be transformed
@tvm.script.ir_module
class MyBmmRelu:
    @T.prim_func(s_tir=True)
    def bmm_relu(A: T.Buffer((16, 128, 128), "float32"),
                 B: T.Buffer((16, 128, 128), "float32"),
                 C: T.Buffer((16, 128, 128), "float32")):
        T.func_attr({"global_symbol": "bmm_relu", "tirx.noalias": True})
        Y = T.sblock_alloc_buffer((16, 128, 128), dtype="float32")
        for b, i, j, k in T.grid(16, 128, 128, 128):
            with T.sblock("Y"):
                vb, vi, vj, vk = T.axis.remap("SSSR", [b, i, j, k])
                with T.init():
                    Y[vb, vi, vj] = T.float32(0.0)
                Y[vb, vi, vj] += A[vb, vi, vk] * B[vb, vk, vj]
        for b, i, j in T.grid(16, 128, 128):
            with T.sblock("C"):
                vb, vi, vj = T.axis.remap("SSS", [b, i, j])
                C[vb, vi, vj] = T.max(Y[vb, vi, vj], T.float32(0.0))

_ = build_and_check(
    MyBmmRelu, "bmm_relu", (bmm_a, bmm_b), bmm_expected, exercise_cpu,
)


bmm_relu: PASS


In [ ]:
def schedule_bmm_relu(mod):
    # WARNING: Must follow the textbook's requirement
    sch = tvm.s_tir.Schedule(mod)
    block_Y = sch.get_sblock("Y", func_name="bmm_relu")
    b, i, j, k = sch.get_loops(block_Y)
    j0, j1 = sch.split(j, factors=[None, 8])
    k0, k1 = sch.split(k, factors=[None, 4])
    sch.reorder(b, i, j0, k0, k1, j1)

    block_C = sch.get_sblock("C", func_name="bmm_relu")
    sch.reverse_compute_at(block_C, j0)

    sch.parallel(b)
    sch.unroll(k1)

    # NOTE: initilization is defaultly elementwise, inside the deepest of loop
    # Now Initialization is put before k0 iterations
    Y_init = sch.decompose_reduction(block_Y, k0)
    sch.vectorize(sch.get_loops(Y_init)[-1])# has shape like [B,I,J,auto]
    sch.vectorize(sch.get_loops(block_C)[-1]) # the same

    return sch.mod

scheduled_bmm = schedule_bmm_relu(MyBmmRelu)
print(scheduled_bmm.script())


# from tvm.script import ir as I
# from tvm.script import tirx as T
# from tvm.tirx.layout import Axis

@I.ir_module
class Module:
    @T.prim_func(s_tir=True)
    def bmm_relu(A: T.Buffer((16, 128, 128), "float32"), B: T.Buffer((16, 128, 128), "float32"), C: T.Buffer((16, 128, 128), "float32")):
        T.func_attr({"tirx.noalias": True})
        # with T.sblock("root"):
        Y = T.sblock_alloc_buffer((16, 128, 128))
        for b in T.parallel(16):
            for i, j_0 in T.grid(128, 16):
                for j_1_init in T.vectorized(8):
                    with T.sblock("Y_init"):
                        vb, vi = T.axis.remap("SS", [b, i])
                        vj = T.axis.spatial(128, j_0 * 8 + j_1_init)
                        T.reads()
                        T.writes(Y[vb, vi, vj])
                        Y[vb, vi, vj] = T.float32(0.0)
                for k_0 in range(32):
                    for k_1 in T.unroll(4):
                        for j_1 in range(8):
            

### 2.5.2.3 Build and Evaluate

Check both kernels against NumPy, then compare median runtime with the same inputs and timer settings.

In [ ]:
def compare_bmm_relu(before, after):
    times_ms = []
    for label, mod in (("Original", before), ("Scheduled", after)):
        lib, args = build_and_check(
            mod, "bmm_relu", (bmm_a, bmm_b), bmm_expected, exercise_cpu,
        )
        timer = lib.time_evaluator("bmm_relu", exercise_cpu, number=10, repeat=5)
        elapsed_ms = float(np.median(timer(*args).results)) * 1_000
        times_ms.append(elapsed_ms)
        print(f"{label:>9}: {elapsed_ms:.3f} ms")
    print(f"Speedup: {times_ms[0] / times_ms[1]:.2f}x")


compare_bmm_relu(MyBmmRelu, scheduled_bmm)


bmm_relu: PASS
 Original: 51.824 ms
bmm_relu: PASS
Scheduled: 6.555 ms
Speedup: 7.91x


## 4. MetaSchedule Practice

Compare the baseline TensorIR, a hand-written schedule, random search, and two MetaSchedule searches on the same 128 × 128 FP32 matrix multiplication. Look for `decision` in each trace and compare the resulting loops in TensorIR.

This section uses a single CPU thread and pinned TVM and FFI versions. Start with the installation cell in a fresh runtime. If an earlier chapter has already imported TVM, restart the session before running this section.

[MLC textbook, sections 4.3–4.5.1](https://book.mlc.ai/chapter_auto_program_optimization/)

In [ ]:
%pip install -q "apache-tvm[popen-pool]==0.27.0.post1" "apache-tvm-ffi==0.1.14.post0" "xgboost-cpu==3.4.1" pandas

In [ ]:
# 4.3 · TensorIR workload and fixed inputs
import os
import tempfile
from pathlib import Path
from statistics import median
from time import perf_counter

os.environ['TVM_NUM_THREADS'] = '1'

import numpy as np
import tvm
import tvm_ffi
from tvm import s_tir
from tvm.s_tir import meta_schedule as ms
from tvm.script import tirx as T

if (tvm.__version__, tvm_ffi.__version__) != ('0.27.0.post1', '0.1.14.post0'):
    raise RuntimeError('Run the installation cell, restart the session, then resume from this cell.')

ch4_target = tvm.target.Target({'kind': 'llvm', 'num-cores': 1})
ch4_device = tvm.cpu(0)
print(f'TVM {tvm.__version__} | FFI {tvm_ffi.__version__} | {ch4_target}')

@tvm.script.ir_module
class Chapter4Matmul:
    @T.prim_func(s_tir=True)
    def main(
        A: T.Buffer((128, 128), 'float32'),
        B: T.Buffer((128, 128), 'float32'),
        C: T.Buffer((128, 128), 'float32'),
    ):
        T.func_attr({'global_symbol': 'main', 'tirx.noalias': True})
        for i, j, k in T.grid(128, 128, 128):
            with T.sblock('C'):
                vi, vj, vk = T.axis.remap('SSR', [i, j, k])
                with T.init():
                    C[vi, vj] = T.float32(0)
                C[vi, vj] = C[vi, vj] + A[vi, vk] * B[vk, vj]

ch4_rng = np.random.default_rng(0)
ch4_a = ch4_rng.standard_normal((128, 128), dtype=np.float32)
ch4_b = ch4_rng.standard_normal((128, 128), dtype=np.float32)
ch4_expected = ch4_a @ ch4_b
ch4_args = (
    tvm.runtime.tensor(ch4_a, device=ch4_device),
    tvm.runtime.tensor(ch4_b, device=ch4_device),
    tvm.runtime.empty(ch4_expected.shape, 'float32', ch4_device),
)

TVM 0.27.0.post1 | FFI 0.1.14.post0 | {"kind":"llvm","tag":"","keys":["cpu"],"mtriple":"x86_64-conda-linux-gnu","num-cores":1}


In [ ]:
# 4.3 · Baseline: compile, verify against NumPy, and benchmark
def ch4_benchmark_ms(mod):
    executable = tvm.compile(mod, target=ch4_target)
    executable.mod['main'](*ch4_args)
    np.testing.assert_allclose(ch4_args[-1].numpy(), ch4_expected, rtol=1e-4, atol=1e-4)
    timer = executable.mod.time_evaluator(
        'main', ch4_device, number=10, repeat=7, min_repeat_ms=10
    )
    return 1_000 * median(timer(*ch4_args).results)

ch4_baseline_ms = ch4_benchmark_ms(Chapter4Matmul)
print(f'Baseline {ch4_baseline_ms:.4f} ms')
print(Chapter4Matmul.script())

Baseline 2.7583 ms
# from tvm.script import ir as I
# from tvm.script import tirx as T
# from tvm.tirx.layout import Axis

@I.ir_module
class Module:
    @T.prim_func(s_tir=True)
    def main(A: T.Buffer((128, 128), "float32"), B: T.Buffer((128, 128), "float32"), C: T.Buffer((128, 128), "float32")):
        T.func_attr({"tirx.noalias": True})
        # with T.sblock("root"):
        for i, j, k in T.grid(128, 128, 128):
            with T.sblock("C"):
                vi, vj, vk = T.axis.remap("SSR", [i, j, k])
                T.reads(A[vi, vk], B[vk, vj])
                T.writes(C[vi, vj])
                with T.init():
                    C[vi, vj] = T.float32(0.0)
                C[vi, vj] = C[vi, vj] + A[vi, vk] * B[vk, vj]


In [ ]:
# 4.3.1 · Fixed schedule: split j, reorder, and separate reduction init
def ch4_schedule_mm(sch, inner_factor=None):
    block = sch.get_sblock('C', func_name='main')
    i, j, k = sch.get_loops(block)
    factors = (
        sch.sample_perfect_tile(j, n=2, max_innermost_factor=16)
        if inner_factor is None
        else [None, inner_factor]
    )
    j_outer, j_inner = sch.split(j, factors=factors)
    sch.reorder(i, j_outer, k, j_inner)
    sch.decompose_reduction(block, k)
    return sch

ch4_fixed = ch4_schedule_mm(s_tir.Schedule(Chapter4Matmul), inner_factor=4)
ch4_fixed_ms = ch4_benchmark_ms(ch4_fixed.mod)
print(f'Hand-written schedule {ch4_fixed_ms:.4f} ms')
print(ch4_fixed.trace)
print(ch4_fixed.mod.script())

Hand-written schedule 0.6733 ms
# from tvm import s_tir
def apply_trace(sch: s_tir.Schedule) -> None:
  b0 = sch.get_sblock(name="C", func_name="main")
  l1, l2, l3 = sch.get_loops(block=b0)
  l4, l5 = sch.split(loop=l2, factors=[None, 4], preserve_unit_iters=True, disable_predication=False)
  sch.reorder(l1, l4, l3, l5)
  b6 = sch.decompose_reduction(block=b0, loop=l3)
# from tvm.script import ir as I
# from tvm.script import tirx as T
# from tvm.tirx.layout import Axis

@I.ir_module
class Module:
    @T.prim_func(s_tir=True)
    def main(A: T.Buffer((128, 128), "float32"), B: T.Buffer((128, 128), "float32"), C: T.Buffer((128, 128), "float32")):
        T.func_attr({"tirx.noalias": True})
        # with T.sblock("root"):
        for i, j_0 in T.grid(128, 32):
            for j_1_init in range(4):
                with T.sblock("C_init"):
                    vi = T.axis.spatial(128, i)
                    vj = T.axis.spatial(128, j_0 * 4 + j_1_init)
                    T.reads()
       

In [ ]:
# 4.4 · Stochastic schedules: compare the decision=[j_outer, j_inner] lines
def ch4_stochastic_schedule(sch):
    return ch4_schedule_mm(sch)

for seed in (11, 12, 13):
    sampled = ch4_stochastic_schedule(s_tir.Schedule(Chapter4Matmul, seed=seed))
    print(f'\nseed={seed}')
    print(sampled.trace)


seed=11
# from tvm import s_tir
def apply_trace(sch: s_tir.Schedule) -> None:
  b0 = sch.get_sblock(name="C", func_name="main")
  l1, l2, l3 = sch.get_loops(block=b0)
  v4, v5 = sch.sample_perfect_tile(loop=l2, n=2, max_innermost_factor=16, decision=[128, 1])
  l6, l7 = sch.split(loop=l2, factors=[v4, v5], preserve_unit_iters=True, disable_predication=False)
  sch.reorder(l1, l6, l3, l7)
  b8 = sch.decompose_reduction(block=b0, loop=l3)

seed=12
# from tvm import s_tir
def apply_trace(sch: s_tir.Schedule) -> None:
  b0 = sch.get_sblock(name="C", func_name="main")
  l1, l2, l3 = sch.get_loops(block=b0)
  v4, v5 = sch.sample_perfect_tile(loop=l2, n=2, max_innermost_factor=16, decision=[32, 4])
  l6, l7 = sch.split(loop=l2, factors=[v4, v5], preserve_unit_iters=True, disable_predication=False)
  sch.reorder(l1, l6, l3, l7)
  b8 = sch.decompose_reduction(block=b0, loop=l3)

seed=13
# from tvm import s_tir
def apply_trace(sch: s_tir.Schedule) -> None:
  b0 = sch.get_sblock(name="C", func_n

In [ ]:
# 4.5 · Manual random search on the same five possible inner tile sizes
ch4_random_trials = []
for seed in range(21, 26):
    candidate = ch4_stochastic_schedule(s_tir.Schedule(Chapter4Matmul, seed=seed))
    elapsed_ms = ch4_benchmark_ms(candidate.mod)
    ch4_random_trials.append((elapsed_ms, candidate))
    tile = next(iter(candidate.trace.decisions.values()))
    print(f'seed={seed} | tile={list(tile)} | {elapsed_ms:.4f} ms')

_, ch4_random_best = min(ch4_random_trials, key=lambda trial: trial[0])
ch4_random_ms = ch4_benchmark_ms(ch4_random_best.mod)
print('\nBest trace from random search')
print(ch4_random_best.trace)

seed=21 | tile=[64, 2] | 1.5475 ms
seed=22 | tile=[64, 2] | 1.5313 ms
seed=23 | tile=[128, 1] | 2.6066 ms
seed=24 | tile=[8, 16] | 0.2697 ms
seed=25 | tile=[32, 4] | 0.7294 ms

Best trace from random search
# from tvm import s_tir
def apply_trace(sch: s_tir.Schedule) -> None:
  b0 = sch.get_sblock(name="C", func_name="main")
  l1, l2, l3 = sch.get_loops(block=b0)
  v4, v5 = sch.sample_perfect_tile(loop=l2, n=2, max_innermost_factor=16, decision=[8, 16])
  l6, l7 = sch.split(loop=l2, factors=[v4, v5], preserve_unit_iters=True, disable_predication=False)
  sch.reorder(l1, l6, l3, l7)
  b8 = sch.decompose_reduction(block=b0, loop=l3)


In [ ]:
# 4.5 · Measure MetaSchedule candidates in the Colab CPU kernel.
# This avoids starting a separate Python worker for each measurement batch.
from tvm.ir.utils import derived_object
from tvm.s_tir.meta_schedule.builder.local_builder import default_build

CH4_CUSTOM_TRIALS = 8
CH4_DEFAULT_TRIALS = 16


@derived_object
class Chapter4Builder(ms.builder.PyBuilder):
    def __init__(self, modules):
        self.modules = modules

    def build(self, inputs):
        results = []
        for i, candidate in enumerate(inputs):
            key = f'memory:{i}'
            try:
                self.modules[key] = default_build(candidate.mod, candidate.target, None)
                results.append(ms.builder.BuilderResult(key, None))
            except Exception as error:
                results.append(ms.builder.BuilderResult(None, str(error)))
        return results


@derived_object
class Chapter4Runner(ms.runner.PyRunner):
    def __init__(self, modules):
        self.modules = modules

    def run(self, inputs):
        futures = []
        for candidate in inputs:
            module = self.modules.pop(str(candidate.artifact_path))
            try:
                timer = module.time_evaluator('main', ch4_device, number=5, repeat=3, min_repeat_ms=5)
                seconds = list(timer(*ch4_args).results)
                result = ms.runner.RunnerResult(seconds, None)
            except Exception as error:
                result = ms.runner.RunnerResult(None, str(error))
            futures.append(ms.runner.RunnerFuture(lambda: True, lambda result=result: result))
        return futures


def ch4_tune(label, trials, **options):
    work_dir = Path(tempfile.mkdtemp(prefix=f'mlc-ch4-{label}-'))
    print(f'Tuning directory {work_dir}', flush=True)
    start = perf_counter()
    modules = {}
    database = ms.tune_tir(
        mod=Chapter4Matmul,
        target=ch4_target,
        work_dir=str(work_dir),
        max_trials_global=trials,
        num_trials_per_iter=min(trials, 8),
        num_tuning_cores=1,
        seed=42,
        builder=Chapter4Builder(modules),
        runner=Chapter4Runner(modules),
        measure_callbacks=[ms.measure_callback.AddToDatabase(), ms.measure_callback.UpdateCostModel()],
        **options,
    )
    elapsed_s = perf_counter() - start
    winner = ms.tir_integration.compile_tir(database, Chapter4Matmul, ch4_target)
    if winner is None:
        raise RuntimeError(f'No usable tuning records. See {work_dir / "logs"}')
    print(f'Tuning time {elapsed_s:.1f} s')
    return database, winner, elapsed_s


def ch4_show_records(database, n=5):
    best = database.query_tuning_record(Chapter4Matmul, ch4_target, workload_name='main')
    records = database.get_top_k(best.workload, n)
    print(f'Database contains {len(database)} records. Mean runtimes for the fastest valid records:')
    for rank, record in enumerate(records, start=1):
        elapsed_ms = 1_000 * np.mean([float(t) for t in record.run_secs])
        print(f'{rank:>2}. {elapsed_ms:.4f} ms')

In [ ]:
# 4.5 · Tune a custom search space using the earlier stochastic schedule
# ReplayTrace resamples its tile decision. XGBoost is introduced in the default-space run.
ch4_custom_db, ch4_custom_best, ch4_custom_tune_s = ch4_tune(
    'custom',
    CH4_CUSTOM_TRIALS,
    space=ms.space_generator.ScheduleFn(ch4_stochastic_schedule),
    strategy='replay-trace',
    cost_model='random',
)

Tuning directory /tmp/mlc-ch4-custom-efzo4cew
2026-11-23 13:35:52 [INFO] Logging directory: /tmp/mlc-ch4-custom-efzo4cew/logs
2026-11-23 13:35:52 [INFO] [task_scheduler.cc:174] Initializing Task #0: "main"
2026-11-23 13:35:52 [INFO] [task_scheduler.cc:344] 
 ID | Name |    FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
------------------------------------------------------------------------------------------------------
  0 | main | 4194304 |      1 |            N/A |          N/A |                   N/A |      0 |      
------------------------------------------------------------------------------------------------------
Total trials: 0
Total latency (us): 0

2026-11-23 13:35:52 [INFO] [task_scheduler.cc:195] TaskScheduler picks Task #0: "main"
2026-11-23 13:35:52 [INFO] [task_scheduler.cc:208] Sending 8 sample(s) to builder
2026-11-23 13:35:52 [INFO] [task_scheduler.cc:218] Sending 8 valid sample(s) to runner
2026-11-23 13:35:53 [INFO] [task_sc

In [ ]:
# 4.5 · Read the fastest record and replay its trace on fresh TensorIR
ch4_show_records(ch4_custom_db)
print('\nBest trace from the custom space')
print(ch4_custom_best.trace)
print(ch4_custom_best.mod.script())

ch4_replayed = s_tir.Schedule(Chapter4Matmul)
ch4_custom_best.trace.apply_to_schedule(ch4_replayed, remove_postproc=False)
tvm.ir.assert_structural_equal(ch4_replayed.mod, ch4_custom_best.mod)
ch4_custom_ms = ch4_benchmark_ms(ch4_replayed.mod)
print(f'Trace replay and NumPy validation passed | {ch4_custom_ms:.4f} ms')

Database contains 8 records. Mean runtimes for the fastest valid records:
 1. 0.2691 ms
 2. 0.2692 ms
 3. 0.2771 ms
 4. 0.2895 ms
 5. 0.3970 ms

Best trace from the custom space
# from tvm import s_tir
def apply_trace(sch: s_tir.Schedule) -> None:
  b0 = sch.get_sblock(name="C", func_name="main")
  l1, l2, l3 = sch.get_loops(block=b0)
  v4, v5 = sch.sample_perfect_tile(loop=l2, n=2, max_innermost_factor=16, decision=[8, 16])
  l6, l7 = sch.split(loop=l2, factors=[v4, v5], preserve_unit_iters=True, disable_predication=False)
  sch.reorder(l1, l6, l3, l7)
  b8 = sch.decompose_reduction(block=b0, loop=l3)
  sch.enter_postproc()
# from tvm.script import ir as I
# from tvm.script import tirx as T
# from tvm.tirx.layout import Axis

@I.ir_module
class Module:
    @T.prim_func(s_tir=True)
    def main(A: T.Buffer((128, 128), "float32"), B: T.Buffer((128, 128), "float32"), C: T.Buffer((128, 128), "float32")):
        T.func_attr({"tirx.noalias": True})
        # with T.sblock("root"):
        

In [ ]:
# 4.5.1 · Default CPU rules and evolutionary search
# A warm-up of 8 allows the second batch to use learned scores in this short demo.
ch4_default_db, ch4_default_best, ch4_default_tune_s = ch4_tune(
    'default',
    CH4_DEFAULT_TRIALS,
    cost_model=ms.cost_model.XGBModel(num_warmup_samples=8, num_tuning_cores=1),
)

Tuning directory /tmp/mlc-ch4-default-ehmd5i23
2026-11-23 13:35:53 [INFO] Logging directory: /tmp/mlc-ch4-default-ehmd5i23/logs
2026-11-23 13:35:53 [INFO] [task_scheduler.cc:174] Initializing Task #0: "main"
2026-11-23 13:35:53 [INFO] [task_scheduler.cc:344] 
 ID | Name |    FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
------------------------------------------------------------------------------------------------------
  0 | main | 4194304 |      1 |            N/A |          N/A |                   N/A |      0 |      
------------------------------------------------------------------------------------------------------
Total trials: 0
Total latency (us): 0

2026-11-23 13:35:53 [INFO] [task_scheduler.cc:195] TaskScheduler picks Task #0: "main"
2026-11-23 13:36:02 [INFO] [task_scheduler.cc:208] Sending 8 sample(s) to builder
2026-11-23 13:36:03 [INFO] [task_scheduler.cc:218] Sending 8 valid sample(s) to runner
2026-11-23 13:36:04 [DEBUG] XGB i

In [ ]:
# 4.5.1 · Inspect the default-space winner and benchmark its TensorIR
ch4_show_records(ch4_default_db)
print('\nBest trace from the default space')
print(ch4_default_best.trace)
print(ch4_default_best.mod.script())
ch4_default_ms = ch4_benchmark_ms(ch4_default_best.mod)
print(f'Default-space winner passed NumPy validation on retest | {ch4_default_ms:.4f} ms')

Database contains 16 records. Mean runtimes for the fastest valid records:
 1. 0.2475 ms
 2. 0.2597 ms
 3. 0.2738 ms
 4. 0.2744 ms
 5. 0.2825 ms

Best trace from the default space
# from tvm import s_tir
def apply_trace(sch: s_tir.Schedule) -> None:
  b0 = sch.get_sblock(name="C", func_name="main")
  b1 = sch.get_sblock(name="root", func_name="main")
  sch.annotate(block_or_loop=b0, ann_key="meta_schedule.tiling_structure", ann_val="SSRSRS")
  l2, l3, l4 = sch.get_loops(block=b0)
  v5, v6, v7, v8 = sch.sample_perfect_tile(loop=l2, n=4, max_innermost_factor=64, decision=[1, 4, 16, 2])
  l9, l10, l11, l12 = sch.split(loop=l2, factors=[v5, v6, v7, v8], preserve_unit_iters=True, disable_predication=False)
  v13, v14, v15, v16 = sch.sample_perfect_tile(loop=l3, n=4, max_innermost_factor=64, decision=[1, 4, 8, 4])
  l17, l18, l19, l20 = sch.split(loop=l3, factors=[v13, v14, v15, v16], preserve_unit_iters=True, disable_predication=False)
  v21, v22 = sch.sample_perfect_tile(loop=l4, n=2, max_

In [ ]:
# Summary: same median benchmark for every method, plus the tuning cost
from IPython.display import Markdown, display

ch4_results = [
    ('Baseline TensorIR', ch4_baseline_ms, None),
    ('Hand-written schedule (inner tile 4)', ch4_fixed_ms, None),
    ('Random search (best sample)', ch4_random_ms, None),
    ('MetaSchedule (custom space)', ch4_custom_ms, ch4_custom_tune_s),
    ('MetaSchedule (default space)', ch4_default_ms, ch4_default_tune_s),
]
ch4_table = [
    '| Method | Runtime (ms) | Speedup | Tuning time (s) |',
    '| :-- | --: | --: | --: |',
]
for label, elapsed_ms, tune_s in ch4_results:
    tune_text = '—' if tune_s is None else f'{tune_s:.1f}'
    ch4_table.append(
        f'| {label} | {elapsed_ms:.4f} | {ch4_baseline_ms / elapsed_ms:.2f}× | {tune_text} |'
    )
display(Markdown('\n'.join(ch4_table)))

| Method | Runtime (ms) | Speedup | Tuning time (s) |
| :-- | --: | --: | --: |
| Baseline TensorIR | 2.7583 | 1.00× | — |
| Hand-written schedule (inner tile 4) | 0.6733 | 4.10× | — |
| Random search (best sample) | 0.2812 | 9.81× | — |
| MetaSchedule (custom space) | 0.5039 | 5.47× | 1.0 |
| MetaSchedule (default space) | 0.2484 | 11.11× | 33.4 |